In [2]:
import anthropic
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
import os


## Set the proper working directory

In [3]:
os.chdir('../classifications')


In [4]:
os.getcwd()

'/home/daniel/code/emestrada_parser/classifications'

## Read the subject csv file with statements

`read_statements` will read the subject csv file with all statements, will separate them by topic and will create a csv file ready for AI to classify them.

In [31]:
def read_statements(file: str) -> None:
    df = pd.read_csv(file)
    subject = list(df["subject"].unique())[0]
    topics = sorted( list(df.topic.unique()) )

    # remove subject column
    df = df.drop("subject", axis=1)

    for t in topics:
        df_topic = df[df.topic == t]
        topic = t.lower().replace(" ", "_")
        df_topic.to_csv(f"./csv_files/{subject}_{topic}_statements_to_AI.csv", sep="|", index=False)
        print(f"Created csv file ({subject}_{topic}_statements.csv) to be processed by AI model.")
    
    print("\n\nAll topics where successfully preprocessed to be fed to AI classifier!")

In [32]:
read_statements("./csv_files/quimica_statements.csv")

Created csv file (química_cantidad_química_statements.csv) to be processed by AI model.
Created csv file (química_conf._electrónica_statements.csv) to be processed by AI model.
Created csv file (química_enlace_químico_statements.csv) to be processed by AI model.
Created csv file (química_equilibrio_químico_statements.csv) to be processed by AI model.
Created csv file (química_formulación_statements.csv) to be processed by AI model.
Created csv file (química_reacciones_redox_statements.csv) to be processed by AI model.
Created csv file (química_reactividad_orgánica_statements.csv) to be processed by AI model.
Created csv file (química_solubilidad_statements.csv) to be processed by AI model.
Created csv file (química_termoquímica_statements.csv) to be processed by AI model.
Created csv file (química_ácido_base_statements.csv) to be processed by AI model.


All topics where successfully preprocessed to be fed to AI classifier!


## API consumption

In [12]:
load_dotenv('../etc/.env')
client = anthropic.Anthropic()

In [33]:

def classify_exercises(topic: str, csv_file: str, instructions_md: str, prompt = "prompt.txt") -> list[str]:
    """
    enunciados: lista de dicts con keys año, convocatoria, ejercicio, texto
    retorna:    lista de dicts con keys año, convocatoria, ejercicio, tema, tipo_ejercicio
    """
    instructions = Path(instructions_md)

    # load markdown file with instructions for the current topic
    system_prompt = instructions.read_text()

    # load the csv file with statements
    csv = Path(csv_file)

    # create content of the prompt:
    content=Path(prompt).read_text().format(topic = topic, csv = Path(csv_file).read_text())

    # create API query
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=2048,
        system=system_prompt,
        messages=[
            {"role": "user", "content": content}
        ]
    )

    # clean the API response
    result = [i for i in response.content[0].text.strip().splitlines()]

    return result

## Process all the csv files
Load the csv files for each topic and feed them to the API

In [34]:
os.getcwd()

'/home/daniel/code/emestrada_parser/classifications'

In [35]:
csv_files = [i.name for i in Path("./csv_files").iterdir() if
             i.name.endswith("to_AI.csv") ]
csv_files

['química_enlace_químico_statements_to_AI.csv',
 'química_solubilidad_statements_to_AI.csv',
 'química_cantidad_química_statements_to_AI.csv',
 'química_conf._electrónica_statements_to_AI.csv',
 'química_equilibrio_químico_statements_to_AI.csv',
 'química_reactividad_orgánica_statements_to_AI.csv',
 'química_reacciones_redox_statements_to_AI.csv',
 'química_formulación_statements_to_AI.csv',
 'química_ácido_base_statements_to_AI.csv',
 'química_termoquímica_statements_to_AI.csv']

In [ ]:
responses = {}
for f in csv_files:
    #get topic
    topic = f.replace("_statements_to_AI", "")
    topic = topic.split("_")
    
    subject = topic[0]
    topic = "_".join(topic[1:])

    response = classify_exercises(topic = topic,
                                  csv_file=f"./csv_files/{f}",
                                  instructions_md=f"./instructions_md/{topic}.md")
    
    # add to responses dictionary
    responses[topic] = response